In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from qdrant_client import models, qdrant_client
from qdrant_client.models import SparseVector
from fastembed import SparseTextEmbedding
from sentence_transformers import SentenceTransformer

s:\my_projects\Arabic_News_Agentic_RAG\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
collection_name = "arabic_news"

# Use relative path from agent directory to the data directory
client = qdrant_client.QdrantClient(path="../data/qdrant_db")

model = SentenceTransformer("aubmindlab/bert-base-arabertv02")
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")

C:\Users\saleen\AppData\Local\Temp\ipykernel_15760\3351440819.py:4: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <arabic_news> contains 30148 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = qdrant_client.QdrantClient(path="../data/qdrant_db")
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2117.34it/s]
[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

In [19]:
def _retrieve(query: str, top_k: int =5, category_filter: str = None):
    dense_vec = model.encode(query, normalize_embeddings=True).tolist()
    sparse_vec = list(sparse_model.embed([query]))[0]
    query_filter = None
    if category_filter:
        query_filter = models.Filter(
            must=[models.FieldCondition(
                key="category",
                match=models.MatchValue(value=category_filter)
            )]
        )
    results = client.query_points(
        collection_name=collection_name,
        prefetch=[
            models.Prefetch(query=dense_vec, using="", limit=20, filter=query_filter),
            models.Prefetch(
                query=SparseVector(
                    indices=sparse_vec.indices.tolist(),
                    values=sparse_vec.values.tolist()
                ),
                using="sparse",
                limit=top_k * 2,
                filter=query_filter
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        query_filter=query_filter,
        limit=top_k
    )
    return results.points

In [15]:
def search_news(query: str)->dict:
    points =_retrieve(query, top_k=5)
    return {
        "tool":"search_news",
        "query": query,
        "results": [
            {
                "id": str(p.id),
                "category": p.payload["category"],
                "text": p.payload["text"],
                "score": p.score
            }
            for p in points
        ]
    }


In [5]:
def summarize_topic(query: str) -> dict:
    points = _retrieve(query, top_k=5)
    combined_content = " ".join([point.payload["text"] for point in points  ])
    return {
        "tool": "summarize_topic",
        "query": query,
        "context": combined_content,
        "source_count": len(points)
    }



In [6]:
def compare_timeline(query: str, category: str = None ) -> dict:
    points = _retrieve(query, top_k=10,category_filter= category)
    return {
        "tool": "compare_timeline",
        "query" : query,
        "results": [
            {"text":point.payload["text"], "category": point.payload["category"]}
            for point in points
        ]
    }


In [7]:
def answer_direct(query:str) ->dict:
    return {
        "tool": "answer_direct",
        "query": query,
        "context": ""

    }



In [16]:
search_news("ما هي آخر الأخبار السياسية؟")


{'tool': 'search_news',
 'query': 'ما هي آخر الأخبار السياسية؟',
 'results': [{'id': '80964f55-70ed-4288-bc6a-fe9377ee1535',
   'category': 'Politics',
   'text': '، الثانية بعد الظهر بتوقيت غرينتش. وتتناول المقابلة التي تجريها الزميلة ريما مكتبي مع جنبلاط الفراغ الرئاسي في لبنان والاتصالات الجارية للتوصل الى انتخاب رئيس للجمهورية. كما تتناول المقابلة الوضع في سوريا والتوقعات بالنسبة للمرحلة المقبلة، والتاثير على لبنان',
   'score': 0.5},
  {'id': 'e5ad8f8e-5f02-4fdf-8043-430a42ec483c',
   'category': 'Politics',
   'text': '. ولعل ابرز الشعارات او الاهداف التي ترفعها الحركة هي رفض صمت الرئيس والحكومة والقوى السياسية عما يحدث على الارض. رفض سيطرة الميليشيات المسلحة على العاصمة وعلى بقية المحافظات. رفض الاتفاقيات السياسية مع الميليشيات المسلحة ما لم تفض الى نزع السلاح. رفض اقتحام معسكرات الجيش والامن ونهب السلاح',
   'score': 0.5},
  {'id': 'a2c183bb-68f2-49f9-a4e6-b1ca210e4816',
   'category': 'Politics',
   'text': 'جنبلاط ضيف حصري على قناة الحدث الثلاثاء الحدث.نت يحل رئيس الحزب الاشت

In [21]:
compare_timeline("ما هي آخر الأخبار السياسية؟", category="Politics")


{'tool': 'compare_timeline',
 'query': 'ما هي آخر الأخبار السياسية؟',
 'results': [{'text': '، الثانية بعد الظهر بتوقيت غرينتش. وتتناول المقابلة التي تجريها الزميلة ريما مكتبي مع جنبلاط الفراغ الرئاسي في لبنان والاتصالات الجارية للتوصل الى انتخاب رئيس للجمهورية. كما تتناول المقابلة الوضع في سوريا والتوقعات بالنسبة للمرحلة المقبلة، والتاثير على لبنان',
   'category': 'Politics'},
  {'text': '. ولعل ابرز الشعارات او الاهداف التي ترفعها الحركة هي رفض صمت الرئيس والحكومة والقوى السياسية عما يحدث على الارض. رفض سيطرة الميليشيات المسلحة على العاصمة وعلى بقية المحافظات. رفض الاتفاقيات السياسية مع الميليشيات المسلحة ما لم تفض الى نزع السلاح. رفض اقتحام معسكرات الجيش والامن ونهب السلاح',
   'category': 'Politics'},
  {'text': 'جنبلاط ضيف حصري على قناة الحدث الثلاثاء الحدث.نت يحل رئيس الحزب الاشتراكي اللبناني وليد جنبلاط غدا الثلاثاء ضيفا على قناة الحدث، في مقابلة حصرية تبث الساعة الخامسة عصرا بتوقيت بيروت',
   'category': 'Politics'},
  {'text': '، والذي رشحت معلومات عن مغادرته جنيف. اما على ال

In [12]:
answer_direct("ما هي آخر الأخبار السياسية؟")


{'tool': 'answer_direct',
 'query': 'ما هي آخر الأخبار السياسية؟',
 'context': ''}

In [ ]:
summarize_topic("ما هي آخر الأخبار السياسية؟")

{'tool': 'summarize_topic',
 'query': 'ما هي آخر الأخبار السياسية؟',
 'context': '، الثانية بعد الظهر بتوقيت غرينتش. وتتناول المقابلة التي تجريها الزميلة ريما مكتبي مع جنبلاط الفراغ الرئاسي في لبنان والاتصالات الجارية للتوصل الى انتخاب رئيس للجمهورية. كما تتناول المقابلة الوضع في سوريا والتوقعات بالنسبة للمرحلة المقبلة، والتاثير على لبنان . ولعل ابرز الشعارات او الاهداف التي ترفعها الحركة هي رفض صمت الرئيس والحكومة والقوى السياسية عما يحدث على الارض. رفض سيطرة الميليشيات المسلحة على العاصمة وعلى بقية المحافظات. رفض الاتفاقيات السياسية مع الميليشيات المسلحة ما لم تفض الى نزع السلاح. رفض اقتحام معسكرات الجيش والامن ونهب السلاح جنبلاط ضيف حصري على قناة الحدث الثلاثاء الحدث.نت يحل رئيس الحزب الاشتراكي اللبناني وليد جنبلاط غدا الثلاثاء ضيفا على قناة الحدث، في مقابلة حصرية تبث الساعة الخامسة عصرا بتوقيت بيروت ، المفوضية الاوروبية الى فعل ما هو صحيح، حيث لا يوجد بتقديرها الدليل الكافي على ما هو اكثر امنا، ويجب استخدامه من المواد البديلة. والمنظمة المذكورة هي اتحاد لمنتجي التعبئة والتغليف في ا

In [20]:
print(search_news("الوضع في مصر"))
print(summarize_topic("الوضع في سوريا"))
print(compare_timeline("الانتخابات اللبنانية", category="Politics"))
print(answer_direct("ما هو التضخم؟"))

{'tool': 'search_news', 'query': 'الوضع في مصر', 'results': [{'id': '5f2b1912-4f71-4a9e-bfbd-404551f3bdb6', 'category': 'Politics', 'text': '. نحن نواجه قوسا من النزاعات', 'score': 0.5}, {'id': '2ec6b4c4-7beb-4b18-b834-97d818dc9d85', 'category': 'Politics', 'text': '، قال ان الوزيرين ناقشا الوضع الامني في مصر والحادث الارهابي الاخير الذي وقع في المنيا.كما اشادت روسيا بالدور المصري في التصدي لظاهرة الارهاب الدولي من خلال عضويتها بمجلس الامن.', 'score': 0.5}, {'id': '6c9211d6-dc9e-49cb-be91-53666f53944b', 'category': 'Medical', 'text': '.في هذه الدراسة', 'score': 0.3333333333333333}, {'id': '4f4b0c76-763b-4965-b4e3-dfbaa9e1296c', 'category': 'Politics', 'text': 'الجزائر تستضيف اجتماعا ثلاثيا لبحث ازمة ليبيا الحدث.نت اعلنت الخارجيةالجزائرية انها بصدد استضافة اجتماع لوزراء خارجية الجزائر و تونس و مصر في يونيوحزيران المقبل، لبحث اخر تطورات الوضع في ليبيا على الصعيدين الامني والسياسي، وفقا لقناة الحدث', 'score': 0.3333333333333333}, {'id': '96dbe9e6-f3fd-43bb-ad97-e1e9f2107e62', 'category': 

In [9]:
search_news("الاقتصاد")

{'tool': 'search_news',
 'query': 'الاقتصاد',
 'results': [{'id': 'ef1e047c-9dd7-48de-9022-bdc441ba79f5',
   'title': None,
   'category': 'Politics',
   'content': None,
   'score': 0.5},
  {'id': '6b52baa9-818f-447e-b101-b54c4f01d61a',
   'title': None,
   'category': 'Politics',
   'content': None,
   'score': 0.5},
  {'id': '9548063f-12c3-4ccc-97fa-90ceb3ab57d3',
   'title': None,
   'category': 'Politics',
   'content': None,
   'score': 0.3333333333333333},
  {'id': '2c1f5400-9cbc-4f6b-964c-f2cec6cf9210',
   'title': None,
   'category': 'Medical',
   'content': None,
   'score': 0.3333333333333333},
  {'id': 'e6116fa7-3e6d-4d74-90a8-c2a9729e4d44',
   'title': None,
   'category': 'Politics',
   'content': None,
   'score': 0.25}]}